In [15]:
!pip install course_utils

ERROR: Could not find a version that satisfies the requirement course_utils (from versions: none)
ERROR: No matching distribution found for course_utils


In [16]:
import course_utils as utils   # unambiguous, immune to site-packages clashes
print(utils.__file__)          # must point at .../your_project/course_utils.py
print([n for n in dir(utils) if not n.startswith('__')])

ModuleNotFoundError: No module named 'course_utils'

In [2]:
import os
import sys
sys.path.append('../..')
import utils

import panel as pn #GUI
import ollama

pn.extension()

In [3]:
def get_completion_from_messages(messages,model='llama3.2:1b',temperature=0,max_tokens=500):
    """Generate response using Ollama's LLaMA 3.2 1b model """
    formatted_messages=[{'role':msg['role'],'content':msg['content']} for msg in messages]

    try:
        response=ollama.chat(model=model,
                            messages=formatted_messages)
        return response['message']['content'].strip() #Ensure clean output
    except Exception as e:
        print("ERROR : Ollama API call failed :",str(e))
        return '[]' #Return empty JSON list on failure

In [12]:
def process_user_message(user_input,all_messages,debug=True):
    delimiter="```"

    #Step 1 : Input vaildation
    if debug:
        print("Step 1 : Input passed basic check.")

    #Step 2: Extract products and categories 
    try:
        category_and_product_response=utils.find_category_and_product_only(
            user_input,utils.get_products_and_category()
        )
        category_and_product_list=utils.read_string_to_list(category_and_product_response)
    except Exception as e:
        category_and_product_list=[]
        if debug:
            print(f"Step 2 Error: Failed to extract categories /products -{e}")

        if debug:
            print(f"Step 2 Error : Failed to extract categories/products - {e}")

    # Step 3 : Fetch product information
    product_information = utils.generate_output_string(category_and_product_list) if category_and_product_list else "No Matching products found."

    if debug:
        print("Step 3 : Retrived product information.")

    # Step 4 : Generate customer service response
    step_4_system_message_content = "You are a helpful AI Assistant . Provide detailed and accurate information. "

    messages = [
        {'role':'system','content':step_4_system_message_content},
        {'role':'user','content':f"{delimiter}{user_input}{delimiter}"},
        {'role':'assistant','content':f"Relevent product information :\n{product_information}"},        
    ]

    all_messages.extend(messages)
    final_response = get_completion_from_messages(all_messages)
    all_messages.append({'role':'assistant','content':final_response})

    if debug:
        print('Step 4 : Generated response')

    #Step 5 : Evaluate response quality

    step_6_system_message_content="Evaluate wheather the AI response fully answers the customer's question. "

    evaluation_message = [
        {'role':'user','content':step_6_system_message_content +
        f"\n\nCustomer message: {delimiter}{user_input}{delimiter}\n"+
        f"Agent response :{delimiter}{final_response}{delimiter}\n"+
        f"Does the response sufficiently answer the question ? Reply in a single word: Y or N"
        }
    ]
    evaluation_response=get_completion_from_messages(evaluation_message)
    cleaned_evaluation_response=evaluation_response.strip().upper()

    if debug:
        print(f"Step 6 : Evaluation result:{cleaned_evaluation_response}")

    #Step 7 : Decision based on evaluation
    if cleaned_evaluation_response=="Y":
        if debug:
            print("Step 7:Response is approved")
        return final_response,all_messages
    elif cleaned_evaluation_response=="N":
        if debug:
            print('Step 7 : Response is not sufficient.')
        return "I'm Unable to provide the information you're looking for . Let me Connect you with a respresntative.",all_messages
    else:
        if debug:
            print(f"Step 7 : Unexpected evaluation response :{cleaned_evaluation_response}")
        return "There was an issue processing your request . Please try again",all_messages

#Test the Function
user_input="Tell me about the SmartX Pro Phone and the FotoSnap Camera , the DSLR one. Also , what tell me about your TVs?"
response,_=process_user_message(user_input,[])
    
print(response)

Step 1 : Input passed basic check.
Step 2 Error: Failed to extract categories /products -module 'utils' has no attribute 'find_category_and_product_only'
Step 2 Error : Failed to extract categories/products - module 'utils' has no attribute 'find_category_and_product_only'
Step 3 : Retrived product information.
Step 4 : Generated response
Step 6 : Evaluation result:Y
Step 7:Response is approved

